# Nuclear Waste Canister Temperature Prediction — Deep Learning
**CIVIL-226 - Introduction to Machine Learning for Engineers — EPFL**
**Team name:** [À remplir]
**Members:** [À remplir]

Ce notebook implémente un réseau de neurones profond inspiré du `ThreeLayerNet` du lab 6,
entraîné avec Adam + backpropagation sur CPU (16GB RAM).

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import warnings
warnings.filterwarnings('ignore')

device = torch.device('cpu')
np.random.seed(42)
torch.manual_seed(42)
print(f'Device : {device}')
print('Imports OK')

## 2. Chargement des données

In [ ]:
sensors = pd.read_parquet('data/sensors.parquet')
train   = pd.read_parquet('data/train.parquet')
test    = pd.read_parquet('data/test.parquet')

# Suppression des capteurs dupliqués (N206, N213 — coordonnées identiques)
sensors = sensors.drop_duplicates(subset='sensor', keep='first').reset_index(drop=True)

print(f'Sensors : {sensors.shape}')
print(f'Train   : {train.shape}')
print(f'Test    : {test.shape}')

## 3. Preprocessing & Feature Engineering

In [ ]:
# Nettoyage : NaN + aberrants filtrés AVANT le split
train_clean = train.dropna(subset=['temperature']).copy()
mask = (train_clean['temperature'] > -10) & (train_clean['temperature'] < 200)
train_clean = train_clean[mask].copy()
print(f'Lignes après nettoyage : {len(train_clean)}')

t_max = train_clean['time'].max()

In [ ]:
def add_features(df, sensors_df, t_max):
    """
    Joint les coordonnées et ajoute des features dérivées.
    Motivées par la physique du transfert thermique :
    - dist_canister : température décroît avec la distance
    - time_log : dynamique exponentielle du transfert
    - is_opa : zone pondérée différemment dans le score Kaggle
    - power_x_time, dist_x_time : interactions physiques
    """
    merged = df.merge(sensors_df, on='sensor', how='left')

    merged['dist_center']   = np.sqrt(merged['coor_x']**2 + merged['coor_y']**2)
    merged['dist_canister'] = np.sqrt((merged['coor_x'] - 0.7)**2 + (merged['coor_y'] - 1.2)**2)
    merged['is_opa']        = (merged['coor_x'] > 1.4).astype(float)
    merged['time_norm']     = merged['time'] / t_max
    merged['time_log']      = np.log1p(merged['time'])
    merged['power_x_time']  = merged['power'] * merged['time_norm']
    merged['dist_x_time']   = merged['dist_canister'] * merged['time_norm']

    return merged

train_feat = add_features(train_clean, sensors, t_max)
test_feat  = add_features(test, sensors, t_max)

FEATURES = ['coor_x', 'coor_y', 'time_norm', 'time_log', 'power',
            'dist_center', 'dist_canister', 'is_opa',
            'power_x_time', 'dist_x_time']
TARGET = 'temperature'

X = train_feat[FEATURES].values
y = train_feat[TARGET].values
print(f'X shape : {X.shape} — {len(FEATURES)} features')

In [ ]:
# Split 80/20 + normalisation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(test_feat[FEATURES].values)

assert len(X_test_s) == len(test)
print(f'Train : {X_train_s.shape} | Val : {X_val_s.shape} | Test : {X_test_s.shape}')

## 4. Architecture du réseau

Inspiré du `ThreeLayerNet` du lab 6, adapté pour la **régression** :
- Couches `Linear` + `ReLU` (comme dans le lab)
- `BatchNorm` pour stabiliser l'entraînement
- `Dropout` pour éviter l'overfitting
- Sortie scalaire — régression, pas classification
- Loss MSE au lieu de CrossEntropy

In [ ]:
class TempNet(nn.Module):
    """
    Réseau de neurones profond pour prédire la température.
    Architecture : 4 couches cachées avec ReLU, BatchNorm et Dropout.
    Inspiré du ThreeLayerNet du lab 6, étendu pour la régression.
    """

    def __init__(self, input_dim: int, hidden_dims: list = [256, 128, 64, 32]):
        super().__init__()

        layers = []
        prev_dim = input_dim

        for hidden_dim in hidden_dims:
            layers += [
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(p=0.2)
            ]
            prev_dim = hidden_dim

        layers.append(nn.Linear(prev_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)

model = TempNet(input_dim=len(FEATURES)).to(device)
print(model)
print(f'\nParamètres entraînables : {sum(p.numel() for p in model.parameters()):,}')

## 5. Préparation des données PyTorch

On utilise un **sample de 500k points** pour que ça tourne en ~10 min sur CPU.
Le DataLoader découpe en mini-batches pour le SGD — exactement comme dans le lab 6.

In [ ]:
# Sample 500k pour CPU (toutes les données = trop lent sans GPU)
np.random.seed(42)
sample_idx = np.random.choice(len(X_train_s), size=500_000, replace=False)
X_train_sample = X_train_s[sample_idx]
y_train_sample = y_train[sample_idx]

X_train_t = torch.tensor(X_train_sample, dtype=torch.float32)
y_train_t = torch.tensor(y_train_sample, dtype=torch.float32).unsqueeze(1)
X_val_t   = torch.tensor(X_val_s,        dtype=torch.float32)
X_test_t  = torch.tensor(X_test_s,       dtype=torch.float32)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader  = DataLoader(train_dataset, batch_size=8192, shuffle=True, num_workers=0)

print(f'Sample train  : {X_train_t.shape}')
print(f'Batches/epoch : {len(train_loader)}')

## 6. Training Loop

Exactement les 5 étapes du lab 6 :
1. `zero_grad()` — reset des gradients
2. Forward pass
3. Compute loss (MSE)
4. `backward()` — backpropagation
5. `optimizer.step()` — mise à jour des poids (Adam)

In [ ]:
def train_epoch(model, loader, loss_fn, optimizer):
    """Entraîne le modèle sur une epoch complète (5 étapes du lab 6)."""
    model.train()
    total_loss = 0.0
    for X_batch, y_batch in loader:
        optimizer.zero_grad()              # 1. reset gradients
        y_pred = model(X_batch)            # 2. forward pass
        loss   = loss_fn(y_pred, y_batch)  # 3. compute loss
        loss.backward()                    # 4. backpropagation
        optimizer.step()                   # 5. update weights
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, X_t, y_np):
    """Calcule le RMSE sur un set donné sans calculer les gradients."""
    model.eval()
    with torch.no_grad():
        y_pred = model(X_t).squeeze().numpy()
    return np.sqrt(mean_squared_error(y_np, y_pred))


print('Fonctions définies OK')

In [ ]:
loss_fn   = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

EPOCHS = 30
train_losses = []
val_rmses    = []
best_rmse    = float('inf')
best_weights = None

print(f'Entraînement sur {EPOCHS} epochs...')
for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, loss_fn, optimizer)
    val_rmse   = evaluate(model, X_val_t, y_val)

    train_losses.append(train_loss)
    val_rmses.append(val_rmse)
    scheduler.step(val_rmse)

    # Sauvegarde du meilleur modèle
    if val_rmse < best_rmse:
        best_rmse = val_rmse
        best_weights = {k: v.clone() for k, v in model.state_dict().items()}

    if epoch % 5 == 0:
        print(f'Epoch {epoch:3d}/{EPOCHS} | Loss : {train_loss:.4f} | Val RMSE : {val_rmse:.4f} | Best : {best_rmse:.4f}')

model.load_state_dict(best_weights)
print(f'\nMeilleur RMSE validation : {best_rmse:.4f}')

In [ ]:
# Courbes d'entraînement
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses, color='steelblue')
axes[0].set_title('MSE Loss (train)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE')

axes[1].plot(val_rmses, color='tomato')
axes[1].axhline(best_rmse, linestyle='--', color='gray', label=f'Best = {best_rmse:.4f}')
axes[1].set_title('RMSE Validation')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('RMSE')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Prédictions finales & Soumission

In [ ]:
model.eval()
with torch.no_grad():
    y_pred = model(X_test_t).squeeze().numpy()

submission = pd.DataFrame({
    'Id': np.arange(len(test), dtype=int),
    'temperature': y_pred.astype(float)
})

assert list(submission.columns) == ['Id', 'temperature']
assert len(submission) == len(test)
assert np.isfinite(submission['temperature']).all()
assert submission.isna().sum().sum() == 0

submission.to_csv('submission.csv', index=False)
print(f'submission.csv sauvegardé — {len(submission)} lignes')
display(submission.head())